# Moment Network — posterior mean + variance per parameter

Follows Jeffrey & Wandelt 2020 (arXiv:2011.05991). For each case:
  1. Train F(x) with MSE — posterior mean μ (already done by test-noise pipeline).
  2. K-fold OOF residuals from F → variance targets v_i = log(r_i²), z-scored.
     (OOF because train_loss < val_loss in this project — real overfitting means
      naive same-set residuals would understate σ.)
  3. Train G(x) on (x_train, v_z) with MSE — G predicts log-variance z-score.
  4. At eval: μ = F(x_val); σ = √exp(G(x_val)·std + mean).

Marginal posterior plots then show how obs1-only vs obs2-only vs both compare
per parameter — the probabilistic version of the R² comparison in test-noise.

In [1]:
import sys
import os
import importlib
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
from sklearn.metrics import r2_score, mean_squared_error
import pandas as pd

base_path = "../src/"
sys.path.append(base_path)
import models
import train
from losses import *
import pipeline
import plots

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Data loading (identical to test-noise-Copy1)

In [3]:
datafilename = "../../DATA/data_L50_TNG_v3.hdf5"

with h5py.File(datafilename, "r") as f:
    Parameters = f["Parameters"][:, :1024].T

logflag = np.array([False, False, True, True, True, True, False, False, False, True, True, False, False, True, False, True, False, True, True, False, False, True, True, True, True, True, True, False, True, False, True, False, False, False, True])
logflag = logflag[:Parameters.shape[1]]
if not np.all(Parameters[:, logflag] > 0):
    raise ValueError("Some values to be logged are non-positive.")
PartiallyLoggedParameters = Parameters.copy()
PartiallyLoggedParameters[:, logflag] = np.log(PartiallyLoggedParameters[:, logflag])
means = PartiallyLoggedParameters.mean(axis=0)
stds = PartiallyLoggedParameters.std(axis=0)
Parameters = (PartiallyLoggedParameters - means) / stds

n_sims = Parameters.shape[0]
with h5py.File(datafilename, "r") as f:
    observable_block = {
        key: torch.from_numpy(f[key][:].T).float()
        for key in sorted(f.keys())
        if key != "Parameters" and f[key].shape[-1] == n_sims
    }

Datasets available:
MBH_Mh_s61
MBH_Mh_s90
Mg_Mh_s61
Mg_Mh_s90
Ms_Mh_s61
Ms_Mh_s90
Parameters
Rs_Ms_s61
Rs_Ms_s90
SFRH
SFRH_100Myr
SFRH_z
SFR_Ms_s61
SFR_Ms_s90
Zs_Ms_s61
Zs_Ms_s90
logMh_s61
logMh_s90
logMs_s61
logMs_s90


## Noise cases — start with the three we need for marginal-posterior comparison

In [4]:
noise_cases = {
    # Reference: obs2 alone (SFR)
    "sfr_clean": {"SFR_Ms_s61": 0.0},
    # Reference: obs1 alone (Ms)
    "ms_clean":  {"Ms_Mh_s61": 0.0},
    # Both observables, both clean
    "sfr_0.0_ms_0.0": {"SFR_Ms_s61": 0.0, "Ms_Mh_s61": 0.0},
    # A noisy pair (optional; comment out to speed up the run)
    "sfr_1.0_ms_1.0": {"SFR_Ms_s61": 1.0, "Ms_Mh_s61": 1.0},
}

all_observables = set()
for case in noise_cases.values():
    all_observables.update(case.keys())
x_raw_dict = {key: observable_block[key].numpy() for key in all_observables}

_sorted_obs = sorted(all_observables)
observable_1, observable_2 = _sorted_obs[0], _sorted_obs[1]
print(f"observable_1 = {observable_1}   observable_2 = {observable_2}")

x_normalized_dict = {k: pipeline.normalize(observable_block[k].numpy()) for k in all_observables}
y = torch.from_numpy(Parameters).float()

Total NaNs: 0
Total Infs: 0
Total NaNs: 0
Total Infs: 0
torch.Size([1024, 162])
torch.Size([1024, 35])


## Hyperparameters & train/val split

In [5]:
output_dim   = y.shape[1]
hidden_dims  = [128, 64]
dropout_rate = 0.2
epochs       = 2000
val_fraction = 0.1
batch_size   = 64

# Moment-network specific
K_FOLDS      = 5
FOLD_EPOCHS  = epochs      # partial-F folds train as thoroughly as the main F
VAR_EPOCHS   = epochs

n_val = int(n_sims * val_fraction)
torch.manual_seed(0); np.random.seed(0)
split_perm = torch.randperm(n_sims)
idx_train = split_perm[:-n_val]
idx_val   = split_perm[-n_val:]
perm = np.random.permutation(len(idx_val))

## Configure pipeline and plots modules

In [6]:
importlib.reload(train); importlib.reload(models); importlib.reload(pipeline); importlib.reload(plots)

pipeline.configure(
    observable_1=observable_1, observable_2=observable_2,
    x_normalized_dict=x_normalized_dict, x_raw_dict=x_raw_dict,
    y=y, idx_val=idx_val, idx_train=idx_train,
    batch_size=batch_size, device=device,
    logflag=logflag, means=means, stds=stds, output_dim=output_dim,
    hidden_dims=hidden_dims, dropout_rate=dropout_rate, epochs=epochs,
    perm=perm,
)

## Train F* per case (mean net — same as test-noise pipeline)

In [7]:
all_results = []
for case_name, selected_observables in noise_cases.items():
    print(f"\n=== Training F* for {case_name} ===")
    input_dim = sum(x_raw_dict[k].shape[1] for k in selected_observables)
    model = models.SimpleMLP(input_dim, hidden_dims, output_dim, dropout_rate).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    criterion = nn.MSELoss()

    train_loader_fn = pipeline.make_train_loader_fn(
        selected_observables, x_normalized_dict, y, idx_train, batch_size)
    val_loader = pipeline.make_val_loader_fn(
        selected_observables, x_normalized_dict, y, idx_val, batch_size)()

    train_losses, val_losses = train.fit_with_epoch_noise(
        model=model, train_loader=None, train_loader_fn=train_loader_fn,
        val_loader=val_loader, optimizer=optimizer, criterion=criterion,
        device=device, epochs=epochs,
    )
    all_results.append({
        "case_name": case_name,
        "selected_observables": selected_observables,
        "model": model,
        "train_losses": train_losses,
        "val_losses": val_losses,
    })

pipeline.configure(all_results=all_results)

<module 'models' from '/Users/nicolasgarcia/Downloads/GAL_SBI/camelsPE/jupyter_notebook/../src/models.py'>

## Diagnostic — R² per case (sanity check before the variance net)

In [8]:
r2_matrix = np.zeros((len(all_results), output_dim))
for ri, r in enumerate(all_results):
    preds, trues = pipeline.get_case_predictions(r, mode="aligned")
    r2_matrix[ri] = r2_score(trues, preds, multioutput="raw_values")

param_names = [f"θ{j}" for j in range(output_dim)]
print(pd.DataFrame(r2_matrix,
                   index=[r["case_name"] for r in all_results],
                   columns=param_names).round(2))

## Fit G (variance head) per case — this is the moment-network step

Each call does K-fold retraining of F on train subsets (for OOF residuals), then
trains G on (x_train, log-variance z-scored). Attaches `var_model`, `var_target_mean`,
`var_target_std` to the result dict.

Time budget: K × fold_epochs mean-net trainings + 1 G training per case.

In [9]:
for r in all_results:
    print(f"\n=== Fitting G for {r['case_name']} ===")
    pipeline.fit_variance_head(
        r,
        models_module=models, train_module=train,
        K=K_FOLDS, fold_epochs=FOLD_EPOCHS, var_epochs=VAR_EPOCHS,
    )

Fold 1


Training:   0%|          | 0/500 [00:00<?, ?iter/s]

[Iter  500] validation loss: 0.7901
Fold 2


Training:   0%|          | 0/500 [00:00<?, ?iter/s]

[Iter  500] validation loss: 0.7576
Fold 3


...

[Iter  500] validation loss: 0.7961
Fold 4


Training:   0%|          | 0/500 [00:00<?, ?iter/s]

[Iter  500] validation loss: 0.7785
Fold 5


Training:   0%|          | 0/500 [00:00<?, ?iter/s]

[Iter  500] validation loss: 0.7826


## Configure plots module (needs the fitted var_models on all_results)

In [ ]:
plots.configure(
    all_results=all_results, output_dim=output_dim,
    observable_1=observable_1, observable_2=observable_2,
    logflag=logflag, means=means, stds=stds,
    x_normalized_dict=x_normalized_dict, y=y, idx_val=idx_val,
    param_names=param_names, noise_cases=noise_cases,
    batch_size=batch_size, device=device, perm=perm,
    r2_matrix=r2_matrix,
)

## Marginal posteriors — the payoff plots

For a single validation sim, overlay p(θ|x) Gaussians from each case. Shows
how the obs1-only, obs2-only, and both-clean models constrain each parameter
differently. `space="log_partial"` = standardized log space (Gaussianity is
honest here for log parameters).

In [ ]:
CASES = list(noise_cases.keys())
FOCUS_PARAMS = ["θ0", "θ1", "θ2", "θ4", "θ7", "θ11"]

# Single-sim, single-parameter comparison
for p in FOCUS_PARAMS[:3]:
    plots.plot_marginal_posterior_1d(p, sim_idx=0, cases=CASES, space="log_partial")
    plt.show()

### Grid: multiple sims × chosen cases, one param per figure

In [ ]:
for p in FOCUS_PARAMS:
    plots.plot_marginal_posterior_grid(p, cases=CASES, n_sims=6, seed=42,
                                        space="log_partial")
    plt.show()

### Population σ per case per parameter (bar chart)

In [ ]:
plots.plot_sigma_by_case_bars(FOCUS_PARAMS, CASES, space="log_partial",
                               reducer="median")
plt.show()

### Calibration diagnostic — pull = (μ − true) / σ
Well-calibrated → histogram matches N(0,1). std >> 1 → over-confident (σ too small).

In [ ]:
plots.plot_pull_distribution(CASES, space="normalized", params=FOCUS_PARAMS)
plt.show()